In [1]:
# -*- coding: utf-8 -*-
"""
Build the Fully vs. Semi automation summary table from your main dataset.

Reads the CSV from:
D:\1_Main_Spreadsheet

Outputs a pandas DataFrame with columns:
Metric | Q1 (Fully) | Median (Fully) | Q3 (Fully) | Q1 (Semi) | Median (Semi) | Q3 (Semi)

Notes
- Uses RQ1_F1 == True to select adopters.
- Uses RQ1_F2 == 'Fully_Auto' vs 'Semi_Auto' to split classes.
- Derives rate metrics safely (avoids divide-by-zero).
- Prefers commits from 'commits_GitAPI' if present, else 'local_commit_count'.
- Prefers stars/forks from common GitHub field names.
"""

from pathlib import Path
import numpy as np
import pandas as pd

# -------- configuration --------
WINDOWS_DIR = Path(r"D:\1_Main_Spreadsheet")
PRIORITY_FILES = [
    # put your newest / canonical names first
    "5.0_Total_Repo.csv",
    "3.2_Total_Repo_MR.csv",
    "3.2_Total_Repo.csv",
    "Total_Repo_MR.csv",
    "Total_Repo.csv",
]
# Optional fallbacks (useful if you test in a notebook with a mounted copy)
FALLBACKS = [
    Path("/mnt/data/5.0_Total_Repo.csv"),
    Path("/mnt/data/3.2_Total_Repo_MR.csv"),
    Path("/mnt/data/3.2_Total_Repo.csv"),
]

# Column candidates (adjust/add if your schema differs)
COLS = {
    "prs": ["pull_requests", "prs", "total_prs", "prs_total"],
    "commits": ["commits_GitAPI", "local_commit_count", "commits", "total_commits"],
    "contributors": ["contributors", "num_contributors", "contributors_total"],
    "age_years": ["repo_age", "age_years", "project_age_years"],
    "size": ["size", "code_size", "loc", "repo_size"],
    "stars": ["stargazers_count", "stars"],
    "forks": ["forks_count", "forks"],
    "class": ["RQ1_F2", "automation_class", "class_f2"],
    "adopter": ["RQ1_F1", "Adopted_Primary", "adopted_primary", "is_adopter"],
}

# -------- helpers --------
def choose_dataset_path() -> Path:
    if WINDOWS_DIR.exists():
        for name in PRIORITY_FILES:
            p = WINDOWS_DIR / name
            if p.exists():
                return p
        # Otherwise, prefer any CSV with "repo" in the name; else first CSV
        csvs = sorted(WINDOWS_DIR.glob("*.csv"))
        for p in csvs:
            if "repo" in p.name.lower():
                return p
        if csvs:
            return csvs[0]
    for p in FALLBACKS:
        if p.exists():
            return p
    raise FileNotFoundError(
        f"Dataset not found in {WINDOWS_DIR} or any fallback. "
        f"Looked for: {PRIORITY_FILES}"
    )

def first_existing(df: pd.DataFrame, candidates, required=True):
    for c in candidates:
        if c in df.columns:
            return c
    if required:
        raise KeyError(f"None of the expected columns found: {candidates}")
    return None

def safe_div(num: pd.Series, den: pd.Series) -> pd.Series:
    """Elementwise division with protection against 0/NaN/inf."""
    num = pd.to_numeric(num, errors="coerce")
    den = pd.to_numeric(den, errors="coerce")
    out = np.where((den > 0) & np.isfinite(den), num / den, np.nan)
    return pd.Series(out, index=num.index)

def q123(series: pd.Series):
    """Return (Q1, Median, Q3) on non-NaN values."""
    s = pd.to_numeric(series, errors="coerce").dropna()
    if s.empty:
        return np.nan, np.nan, np.nan
    return (np.quantile(s, 0.25), np.quantile(s, 0.50), np.quantile(s, 0.75))

def fmt(x):
    """Pretty formatting: integers without .0, else two decimals."""
    if pd.isna(x):
        return ""
    if abs(x - round(x)) < 1e-9:
        return str(int(round(x)))
    return f"{x:.2f}"

# -------- load --------
csv_path = choose_dataset_path()
df = pd.read_csv(csv_path, encoding="utf-8-sig")

adopter_col = first_existing(df, COLS["adopter"])
class_col = first_existing(df, COLS["class"])
prs_col = first_existing(df, COLS["prs"])
commits_col = first_existing(df, COLS["commits"])
contributors_col = first_existing(df, COLS["contributors"])
age_col = first_existing(df, COLS["age_years"])
size_col = first_existing(df, COLS["size"], required=False)
stars_col = first_existing(df, COLS["stars"], required=False)
forks_col = first_existing(df, COLS["forks"], required=False)

# -------- filter adopters and split classes --------
# Accept booleans or truthy strings for RQ1_F1
adopter_vals = df[adopter_col]
if adopter_vals.dtype == bool:
    adopters_mask = adopter_vals
else:
    adopters_mask = adopter_vals.astype(str).str.lower().isin(["true", "1", "yes", "adopter"])

adopters = df[adopters_mask].copy()

# Normalize class labels (map to "Fully" vs "Semi", keep others untouched)
class_map = {
    "Fully_Auto": "Fully",
    "Fully": "Fully",
    "Semi_Auto": "Semi",
    "Semi": "Semi",
    "CI_Only": "CI_Only",
}
adopters["Class"] = adopters[class_col].astype(str).map(class_map).fillna(adopters[class_col].astype(str))

fully = adopters[adopters["Class"] == "Fully"].copy()
semi = adopters[adopters["Class"] == "Semi"].copy()

# -------- derive metrics --------
prs_f, prs_s = fully[prs_col], semi[prs_col]
comm_f, comm_s = fully[commits_col], semi[commits_col]
contrib_f, contrib_s = fully[contributors_col], semi[contributors_col]
age_f, age_s = fully[age_col], semi[age_col]

# per-year
prs_per_year_f = safe_div(prs_f, age_f)
prs_per_year_s = safe_div(prs_s, age_s)
comm_per_year_f = safe_div(comm_f, age_f)
comm_per_year_s = safe_div(comm_s, age_s)

# per-contributor
prs_per_contrib_f = safe_div(prs_f, contrib_f)
prs_per_contrib_s = safe_div(prs_s, contrib_s)
comm_per_contrib_f = safe_div(comm_f, contrib_f)
comm_per_contrib_s = safe_div(comm_s, contrib_s)

# commits per PR (granularity)
comm_per_pr_f = safe_div(comm_f, prs_f)
comm_per_pr_s = safe_div(comm_s, prs_s)

# optional fields
size_f = fully[size_col] if size_col else pd.Series(dtype=float)
size_s = semi[size_col] if size_col else pd.Series(dtype=float)
stars_f = fully[stars_col] if stars_col else pd.Series(dtype=float)
stars_s = semi[stars_col] if stars_col else pd.Series(dtype=float)
forks_f = fully[forks_col] if forks_col else pd.Series(dtype=float)
forks_s = semi[forks_col] if forks_col else pd.Series(dtype=float)

# -------- assemble Q1/Median/Q3 table --------
rows = [
    ("PRs/Year",            *q123(prs_per_year_f),        *q123(prs_per_year_s)),
    ("Commits/Year",        *q123(comm_per_year_f),       *q123(comm_per_year_s)),
    ("Commits/PR",          *q123(comm_per_pr_f),         *q123(comm_per_pr_s)),
    ("PRs/Contributor",     *q123(prs_per_contrib_f),     *q123(prs_per_contrib_s)),
    ("Commits/Contributor", *q123(comm_per_contrib_f),    *q123(comm_per_contrib_s)),
    ("PRs",                 *q123(prs_f),                 *q123(prs_s)),
    ("Commits",             *q123(comm_f),                *q123(comm_s)),
    ("Contributors",        *q123(contrib_f),             *q123(contrib_s)),
    ("AgeYears",            *q123(age_f),                 *q123(age_s)),
    ("Size",                *q123(size_f),                *q123(size_s)),
    ("Stars",               *q123(stars_f),               *q123(stars_s)),
    ("Forks",               *q123(forks_f),               *q123(forks_s)),
]

out = pd.DataFrame(
    rows,
    columns=[
        "Metric",
        "Q1 (Fully)", "Median (Fully)", "Q3 (Fully)",
        "Q1 (Semi)",  "Median (Semi)",  "Q3 (Semi)"
    ],
)

# Pretty-format numbers as strings (to mirror your example)
for col in out.columns[1:]:
    out[col] = out[col].map(fmt)

print(f"Dataset used: {csv_path}")
print(f"Fully n={len(fully)}, Semi n={len(semi)}")
print(out.to_string(index=False))

# -------- save next to the dataset --------
out_path = WINDOWS_DIR / "obs2_fully_vs_semi_table.csv"
out.to_csv(out_path, index=False, encoding="utf-8-sig")
print(f"\nSaved: {out_path}")


Dataset used: D:\1_Main_Spreadsheet\5.0_Total_Repo.csv
Fully n=315, Semi n=64
             Metric Q1 (Fully) Median (Fully) Q3 (Fully) Q1 (Semi) Median (Semi) Q3 (Semi)
           PRs/Year       1.86          10.91      58.14      1.58         12.33     74.30
       Commits/Year      18.82          64.41     221.34     26.73         89.49    301.89
         Commits/PR       2.21           5.42      15.77      2.92          6.36     15.35
    PRs/Contributor       2.52           7.27         19      2.38          6.50     18.64
Commits/Contributor      24.89          45.12      96.38     21.40         54.65    139.59
                PRs         11             69        364      8.75            40    281.25
            Commits     132.50            409    1413.50       117        383.50      1299
       Contributors          2              8      31.50      2.75             6     20.50
           AgeYears       4.80           7.64       9.93      3.00          5.70      9.42
            